# Statistical Exchange of Space and Time

Computer algorithms tend to solve the same problem over a spectrum of space-time trade-offs. 
It's reasonable to expect the same from statistical estimation. 
However, my below findings really suggest otherwise. 

In [1]:
## Dense net code initially authored by Google's search engine GenAI on 20 Oct 2024. 
## I've applied minor modifications for generality, but the code worked great on first draft. 

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

DIM = 32 

## define control 
class DenseNetControl(nn.Module):
    def __init__(self):
        super(DenseNetControl, self).__init__()
        self.features = nn.Sequential(
            nn.Linear(784, DIM),
            nn.ReLU(),
            nn.Linear(DIM, DIM),
            nn.ReLU(),
            nn.Linear(DIM, DIM),
            nn.ReLU(),
            nn.Linear(DIM, 10)
        )
    def forward(self, x):
        if x.shape[0] == 0: 
            return torch.tensor([])
        x = x.view(x.size(0), -1)
        x = self.features(x)
        return x
    pass 

## define experimental model 
class DenseNetExperimental(nn.Module):
    def __init__(self):
        super(DenseNetExperimental, self).__init__() 
        self.linear = nn.Linear(DIM, DIM)
        self.features = nn.Sequential(
            nn.Linear(784, DIM),
            nn.ReLU(),
            self.linear,
            nn.ReLU(),
            self.linear,
            nn.ReLU(),
            self.linear,
            nn.ReLU(),
            self.linear,
            nn.ReLU(),
            self.linear,
            nn.ReLU(),
            self.linear,
            nn.ReLU(),
            self.linear,
            nn.ReLU(),
            self.linear,
            nn.ReLU(),
            nn.Linear(DIM, 10)
        )
    def forward(self, x):
        if x.shape[0] == 0: 
            return torch.tensor([])
        x = x.view(x.size(0), -1)
        x = self.features(x)
        return x
    pass 

## Load MNIST dataset
train_dataset = datasets.MNIST(root='/tmp/data', train=True, transform=transforms.ToTensor(), download=True)
test_dataset = datasets.MNIST(root='/tmp/data', train=False, transform=transforms.ToTensor())

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [2]:
## Dense net experiment, needs about 1-2 min of compute 

# Initialize the model, loss function and optimizer 
model_control = DenseNetControl() 
criterion = nn.CrossEntropyLoss() 
model_control.optimizer = optim.Adam(model_control.parameters(), lr=0.001) 

model_experimental = DenseNetExperimental() 
criterion = nn.CrossEntropyLoss() 
model_experimental.optimizer = optim.Adam(model_experimental.parameters(), lr=0.001) 

# Train the model 
def fit(model):
    model.train() 
    num_epochs = 5 
    for epoch in range(num_epochs): 
        for i, (data, target) in enumerate(train_loader): 
            model.optimizer.zero_grad() 
            output = model(data) 
            loss = criterion(output, target) 
            loss.backward() 
            model.optimizer.step() 
            if i % 100 == 0:
                print('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'.format(epoch+1, num_epochs, i+1, len(train_loader), loss.item()))
    # Evaluate the model
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    acc = correct / total
    print('Accuracy of the network on the 10000 test images: {} %'.format(100 * correct / total)) 
    return acc 

In [3]:
acc = fit(model_control)
print(acc)

Epoch [1/5], Step [1/938], Loss: 2.3103
Epoch [1/5], Step [101/938], Loss: 0.9080
Epoch [1/5], Step [201/938], Loss: 0.3622
Epoch [1/5], Step [301/938], Loss: 0.5143
Epoch [1/5], Step [401/938], Loss: 0.3184
Epoch [1/5], Step [501/938], Loss: 0.4292
Epoch [1/5], Step [601/938], Loss: 0.3460
Epoch [1/5], Step [701/938], Loss: 0.2687
Epoch [1/5], Step [801/938], Loss: 0.1678
Epoch [1/5], Step [901/938], Loss: 0.3403
Epoch [2/5], Step [1/938], Loss: 0.2590
Epoch [2/5], Step [101/938], Loss: 0.1689
Epoch [2/5], Step [201/938], Loss: 0.2185
Epoch [2/5], Step [301/938], Loss: 0.1131
Epoch [2/5], Step [401/938], Loss: 0.1632
Epoch [2/5], Step [501/938], Loss: 0.2662
Epoch [2/5], Step [601/938], Loss: 0.2189
Epoch [2/5], Step [701/938], Loss: 0.1241
Epoch [2/5], Step [801/938], Loss: 0.3120
Epoch [2/5], Step [901/938], Loss: 0.3764
Epoch [3/5], Step [1/938], Loss: 0.1428
Epoch [3/5], Step [101/938], Loss: 0.1377
Epoch [3/5], Step [201/938], Loss: 0.0590
Epoch [3/5], Step [301/938], Loss: 0.178

In [4]:
acc = fit(model_experimental)
print(acc)

Epoch [1/5], Step [1/938], Loss: 2.3081
Epoch [1/5], Step [101/938], Loss: 1.5580
Epoch [1/5], Step [201/938], Loss: 0.9096
Epoch [1/5], Step [301/938], Loss: 0.8364
Epoch [1/5], Step [401/938], Loss: 0.7987
Epoch [1/5], Step [501/938], Loss: 0.9808
Epoch [1/5], Step [601/938], Loss: 0.5834
Epoch [1/5], Step [701/938], Loss: 0.2913
Epoch [1/5], Step [801/938], Loss: 0.4463
Epoch [1/5], Step [901/938], Loss: 0.3881
Epoch [2/5], Step [1/938], Loss: 0.5409
Epoch [2/5], Step [101/938], Loss: 0.5749
Epoch [2/5], Step [201/938], Loss: 0.4136
Epoch [2/5], Step [301/938], Loss: 0.4632
Epoch [2/5], Step [401/938], Loss: 0.3608
Epoch [2/5], Step [501/938], Loss: 0.3600
Epoch [2/5], Step [601/938], Loss: 0.1557
Epoch [2/5], Step [701/938], Loss: 0.3553
Epoch [2/5], Step [801/938], Loss: 0.5613
Epoch [2/5], Step [901/938], Loss: 0.2718
Epoch [3/5], Step [1/938], Loss: 0.3478
Epoch [3/5], Step [101/938], Loss: 0.3240
Epoch [3/5], Step [201/938], Loss: 0.3331
Epoch [3/5], Step [301/938], Loss: 0.192